## Why V2 exists

The V2 representation expands the original count-based features with POS composition, lexical-diversity variants, repeated words and phrases, filler density, dependency depth, clause-like structure, conjunction density, and sentence complexity. It is an experiment, not an assumption that the expanded representation must be better.

The reusable implementation lives in `scripts/pipeline_common.py`; this notebook builds and inspects the cached V2 table only. Later experiment results decide whether V2 is selected.

# 11. Linguistic features V2
Build POS, lexical, repetition, disfluency, and sentence-complexity features from cached Whisper transcripts.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.pipeline_common import OUTPUTS, build_linguistic_v2
transcripts = pd.read_csv(OUTPUTS / 'train_transcripts.csv')
features = build_linguistic_v2(transcripts)
features.to_csv(OUTPUTS / 'linguistic_features_v2.csv', index=False)
print(features.shape, OUTPUTS / 'linguistic_features_v2.csv')

In [ ]:
features.head()

In [ ]:
# Summarize the generated V2 table and inspect target correlations.
# The feature construction remains in scripts/pipeline_common.py.
import matplotlib.pyplot as plt

v2_numeric = features.select_dtypes(include=["number"]).drop(columns=["label"], errors="ignore")
v2_correlations = v2_numeric.join(features["label"]).corr(numeric_only=True)["label"].drop("label").sort_values()
print("V2 feature count:", v2_numeric.shape[1])
print(v2_correlations.to_string())

v2_correlations.plot.barh(figsize=(9, max(5, len(v2_correlations) * 0.22)), title="V2 feature correlation with grammar score")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## Interpretation

V2 is a broader experimental representation. Its feature analysis is descriptive; the model-selection notebook, using the fixed cross-validation protocol, decides whether these additions improve generalization. The final selected feature set is documented separately and should not be inferred from this exploratory table alone.